# W7D5 — Multi-document RAG System

## Objective
Build a multi-document Retrieval-Augmented Generation (RAG) system using LangGraph, Ollama, and ChromaDB.

## Components
- **Document ingestion:** Load multiple documents.
- **Embeddings:** Convert document chunks into vectors.
- **Vector database:** Store and retrieve relevant chunks using ChromaDB.
- **RAG workflow:** Use LangGraph to coordinate retrieval and answer generation.
- **Experiment tracking:** Track runs using MLflow.
- **Evaluation:** Evaluate retrieval and answer quality using Ragas where compatible.

## Expected Outcome
A working local RAG pipeline that answers questions using multiple source documents, records its outputs, and includes test and evaluation evidence.

In [1]:
# W7D5: Import required libraries

import os
import time
import mlflow
import chromadb

from pathlib import Path
from langchain_ollama import OllamaLLM, OllamaEmbeddings
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langgraph.graph import StateGraph, START, END

print("Core libraries imported successfully!")

Core libraries imported successfully!


In [3]:
# Use a local SQLite database for MLflow tracking
mlflow.set_tracking_uri(f"sqlite:///{(OUTPUT_DIR / 'mlflow.db').resolve().as_posix()}")

In [4]:
mlflow.set_experiment("W7D5_Multi_Document_RAG")

2026/10/10 10:40:49 INFO mlflow.store.db.utils: Creating initial MLflow database tables...
2026/10/10 10:40:49 INFO mlflow.store.db.utils: Updating database tables
2026/10/10 10:40:55 INFO mlflow.tracking.fluent: Experiment with name 'W7D5_Multi_Document_RAG' does not exist. Creating a new experiment.


<Experiment: artifact_location='file:c:/Users/ughan/CynarisInternship/AIML_PROJECT/notebooks/mlruns/1', creation_time=1791609055628, effective_trace_archival_retention=None, experiment_id='1', last_update_time=1791609055628, lifecycle_stage='active', name='W7D5_Multi_Document_RAG', tags={}, trace_location=None, workspace='default'>

In [5]:
# W7D5: Create sample documents

documents = {
    "python.txt": """
Python is a programming language used in automation, data science,
machine learning, and artificial intelligence.
""",
    "rag.txt": """
Retrieval-Augmented Generation (RAG) retrieves relevant documents
and uses their information to generate context-based answers.
""",
    "mlops.txt": """
MLOps combines machine learning, software engineering, and operations.
It helps track experiments, evaluate models, and manage deployments.
"""
}

for filename, content in documents.items():
    (DATA_DIR / filename).write_text(content.strip(), encoding="utf-8")

print(f"Created {len(documents)} sample documents.")
print("Files:", ", ".join(documents.keys()))

Created 3 sample documents.
Files: python.txt, rag.txt, mlops.txt


In [6]:
# W7D5: Load documents from files

loaded_documents = []

for file_path in DATA_DIR.glob("*.txt"):
    loaded_documents.append({
        "source": file_path.name,
        "text": file_path.read_text(encoding="utf-8")
    })

print("Documents loaded:", len(loaded_documents))

for document in loaded_documents:
    print("-", document["source"])

Documents loaded: 3
- mlops.txt
- python.txt
- rag.txt


In [1]:
# W7D5: Load and split documents

from pathlib import Path
from langchain_text_splitters import RecursiveCharacterTextSplitter

DATA_DIR = Path("notebooks/W7D5/documents")

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=200,
    chunk_overlap=30
)

chunks = []

for file_path in sorted(DATA_DIR.glob("*.txt")):
    text = file_path.read_text(encoding="utf-8")

    for chunk in text_splitter.split_text(text):
        chunks.append({
            "source": file_path.name,
            "text": chunk
        })

print("Documents processed:", len(list(DATA_DIR.glob("*.txt"))))
print("Total chunks:", len(chunks))
print("First chunk:", chunks[0]["text"])

Documents processed: 3
Total chunks: 3
First chunk: MLOps combines machine learning, software engineering, and operations.
It helps track experiments, evaluate models, and manage deployments.


In [2]:
# W7D5: Initialize Ollama embeddings

from langchain_ollama import OllamaEmbeddings

embeddings = OllamaEmbeddings(model="nomic-embed-text")

# Test the embedding model
test_embedding = embeddings.embed_query("What is RAG?")

print("Embedding model: nomic-embed-text")
print("Embedding dimensions:", len(test_embedding))
print("Embedding test successful!")

Embedding model: nomic-embed-text
Embedding dimensions: 768
Embedding test successful!


In [3]:
# W7D5: Store document chunks in ChromaDB

import chromadb

client = chromadb.PersistentClient(
    path="notebooks/W7D5/chroma_db"
)

collection = client.get_or_create_collection(
    name="w7d5_documents"
)

for index, chunk in enumerate(chunks):
    collection.upsert(
        ids=[str(index)],
        documents=[chunk["text"]],
        metadatas=[{"source": chunk["source"]}]
    )

print("Vector store created successfully!")
print("Stored chunks:", collection.count())

Vector store created successfully!
Stored chunks: 3


In [5]:
# W7D5: Recreate ChromaDB collection with correct embeddings

client.delete_collection("w7d5_documents")

collection = client.create_collection(
    name="w7d5_documents",
    metadata={"hnsw:space": "cosine"}
)

for index, chunk in enumerate(chunks):
    collection.upsert(
        ids=[str(index)],
        embeddings=[embeddings.embed_query(chunk["text"])],
        documents=[chunk["text"]],
        metadatas=[{"source": chunk["source"]}]
    )

print("Vector store rebuilt successfully!")
print("Stored chunks:", collection.count())

Vector store rebuilt successfully!
Stored chunks: 3


In [6]:
# W7D5: Retrieve relevant document chunks

query = "What is RAG?"

results = collection.query(
    query_embeddings=[embeddings.embed_query(query)],
    n_results=2
)

for source, text in zip(
    results["metadatas"][0],
    results["documents"][0]
):
    print("Source:", source["source"])
    print("Content:", text)
    print("-" * 40)

Source: rag.txt
Content: Retrieval-Augmented Generation (RAG) retrieves relevant documents
and uses their information to generate context-based answers.
----------------------------------------
Source: python.txt
Content: Python is a programming language used in automation, data science,
machine learning, and artificial intelligence.
----------------------------------------


In [7]:
# W7D5: Define the RAG workflow

from typing import TypedDict
from langgraph.graph import StateGraph, START, END

class RAGState(TypedDict):
    question: str
    context: str

def retrieve_documents(state: RAGState):
    results = collection.query(
        query_embeddings=[
            embeddings.embed_query(state["question"])
        ],
        n_results=2
    )

    context = "\n".join(results["documents"][0])

    return {"context": context}

workflow = StateGraph(RAGState)

workflow.add_node("retrieve", retrieve_documents)
workflow.add_edge(START, "retrieve")
workflow.add_edge("retrieve", END)

rag_app = workflow.compile()

print("LangGraph retrieval workflow created successfully!")

LangGraph retrieval workflow created successfully!


In [8]:
# W7D5: Test the LangGraph workflow

question = "What is RAG?"

result = rag_app.invoke({
    "question": question,
    "context": ""
})

print("Question:", question)
print("\nRetrieved context:")
print(result["context"])

Question: What is RAG?

Retrieved context:
Retrieval-Augmented Generation (RAG) retrieves relevant documents
and uses their information to generate context-based answers.
Python is a programming language used in automation, data science,
machine learning, and artificial intelligence.


In [9]:
# W7D5: Generate an answer from retrieved context

from langchain_ollama import OllamaLLM

llm = OllamaLLM(model="qwen2.5:3b")

context = result["context"]

prompt = f"""
Answer the question using only the context below.

Context:
{context}

Question: {question}

Answer:
"""

answer = llm.invoke(prompt)

print("Question:", question)
print("\nGenerated answer:")
print(answer)

Question: What is RAG?

Generated answer:
Retrieval-Augmented Generation (RAG) is a technique that combines the capabilities of document retrieval systems and natural language generation. It works by utilizing a document retrieval system to find relevant documents that can provide context and information to answer a question. Once the relevant documents are identified, RAG uses the information from these documents to generate a context-based answer to the question.


In [10]:
# W7D5: Log the RAG experiment with MLflow

import mlflow

with mlflow.start_run(run_name="W7D5_RAG_Test"):
    mlflow.log_param("llm_model", "qwen2.5:3b")
    mlflow.log_param("embedding_model", "nomic-embed-text")
    mlflow.log_param("document_count", 3)
    mlflow.log_metric("retrieved_chunks", len(result["context"].split("\n")))

    mlflow.log_text(question, "question.txt")
    mlflow.log_text(answer, "generated_answer.txt")

print("MLflow experiment logged successfully!")

2026/10/10 11:12:06 INFO mlflow.store.db.utils: Creating initial MLflow database tables...
2026/10/10 11:12:06 INFO mlflow.store.db.utils: Updating database tables


MLflow experiment logged successfully!


In [11]:
# W7D5: Evaluate the generated answer

expected_keywords = ["retrieval", "documents", "context"]

answer_lower = answer.lower()

matched_keywords = [
    word for word in expected_keywords
    if word in answer_lower
]

score = len(matched_keywords) / len(expected_keywords)

print("Expected keywords:", expected_keywords)
print("Matched keywords:", matched_keywords)
print(f"Keyword match score: {score:.2f}")
print("Evaluation completed!")

Expected keywords: ['retrieval', 'documents', 'context']
Matched keywords: ['retrieval', 'documents', 'context']
Keyword match score: 1.00
Evaluation completed!


In [13]:
# W7D5: Save evaluation results

from pathlib import Path

OUTPUT_DIR = Path("notebooks/W7D5/outputs")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

evaluation_text = f"""
W7D5 RAG Evaluation

Question: {question}
Expected keywords: {expected_keywords}
Matched keywords: {matched_keywords}
Keyword match score: {score:.2f}

Generated answer:
{answer}
"""

evaluation_file = OUTPUT_DIR / "evaluation_results.txt"
evaluation_file.write_text(evaluation_text.strip(), encoding="utf-8")

print("Evaluation results saved successfully!")
print("File:", evaluation_file)

Evaluation results saved successfully!
File: notebooks\W7D5\outputs\evaluation_results.txt


In [14]:
# W7D5: Run basic pipeline tests

assert len(chunks) == 3, "Document chunk count is incorrect"
assert collection.count() == 3, "Vector store count is incorrect"
assert result["context"], "Retrieved context is empty"
assert answer.strip(), "Generated answer is empty"
assert score == 1.0, "Keyword evaluation failed"

print("All 5 RAG pipeline tests passed!")

All 5 RAG pipeline tests passed!


In [15]:
# W7D5: Save test evidence

test_results = {
    "document_count": len(chunks),
    "stored_chunks": collection.count(),
    "retrieval_working": bool(result["context"]),
    "answer_generation_working": bool(answer.strip()),
    "keyword_score": score,
    "tests_passed": 5
}

test_file = OUTPUT_DIR / "test_results.txt"

with test_file.open("w", encoding="utf-8") as file:
    for name, value in test_results.items():
        file.write(f"{name}: {value}\n")

print("Test evidence saved successfully!")
print("File:", test_file)

Test evidence saved successfully!
File: notebooks\W7D5\outputs\test_results.txt


In [16]:
# W7D5: Log a second RAG interaction

question_2 = "What is MLOps?"

result_2 = rag_app.invoke({
    "question": question_2,
    "context": ""
})

prompt_2 = f"""
Answer using only the context below.

Context:
{result_2["context"]}

Question: {question_2}

Answer:
"""

answer_2 = llm.invoke(prompt_2)

with mlflow.start_run(run_name="W7D5_RAG_Interaction_2"):
    mlflow.log_param("llm_model", "qwen2.5:3b")
    mlflow.log_text(question_2, "question.txt")
    mlflow.log_text(answer_2, "generated_answer.txt")

print("Interaction 2 logged successfully!")
print("Question:", question_2)
print("Answer:", answer_2)

Interaction 2 logged successfully!
Question: What is MLOps?
Answer: MLOps is a term that combines machine learning, software engineering, and operations to streamline the process of managing machine learning models and experiments. It focuses on automating and automating the deployment of machine learning models into production environments, ensuring that experiments can be tracked, evaluated, and models managed efficiently.


In [17]:
# W7D5: Save interaction evidence

interaction_file = OUTPUT_DIR / "interaction_evidence.txt"

interaction_text = f"""
W7D5 RAG Interaction Evidence

Interaction 1
Question: {question}
Answer: {answer}

Interaction 2
Question: {question_2}
Answer: {answer_2}
"""

interaction_file.write_text(
    interaction_text.strip(),
    encoding="utf-8"
)

print("Both interactions saved successfully!")
print("Interactions recorded: 2")
print("File:", interaction_file)

Both interactions saved successfully!
Interactions recorded: 2
File: notebooks\W7D5\outputs\interaction_evidence.txt


In [18]:
# W7D5: Verify saved evidence files

required_files = [
    "evaluation_results.txt",
    "test_results.txt",
    "interaction_evidence.txt"
]

for filename in required_files:
    file_path = OUTPUT_DIR / filename
    print(f"{filename}: {'OK' if file_path.exists() else 'MISSING'}")

print("Evidence verification completed!")

evaluation_results.txt: OK
test_results.txt: OK
interaction_evidence.txt: OK
Evidence verification completed!


In [2]:
# W7D5: Create the self-review checklist

from pathlib import Path

review = """W7D5 — MULTI-DOCUMENT RAG SELF-REVIEW

[✓] Created and loaded three sample documents
[✓] Split documents into chunks
[✓] Generated 768-dimensional Ollama embeddings
[✓] Stored and retrieved chunks using ChromaDB
[✓] Built the retrieval workflow using LangGraph
[✓] Generated answers using Ollama qwen2.5:3b
[✓] Logged experiment data using MLflow
[✓] Evaluated the answer using keyword matching
[✓] Passed five basic pipeline tests
[✓] Saved evaluation and test evidence
[✓] Logged two RAG interactions

Pending:
[ ] Complete Ragas evaluation
[ ] Final verification and Git workflow
"""

review_file = Path("notebooks/W7D5/SELF_REVIEW.md")
review_file.write_text(review, encoding="utf-8")

print("Self-review checklist saved successfully!")
print("File:", review_file)

Self-review checklist saved successfully!
File: notebooks\W7D5\SELF_REVIEW.md


In [3]:
# W7D5: Check Ragas availability

try:
    import ragas
    print("Ragas is available!")
    print("Version:", ragas.__version__)
except Exception as error:
    print("Ragas import failed:")
    print(type(error).__name__, "-", error)

Ragas import failed:
ModuleNotFoundError - No module named 'langchain_community.chat_models.vertexai'


In [4]:
# W7D5: Check Ragas and LangChain package versions

from importlib.metadata import version, PackageNotFoundError

packages = [
    "ragas",
    "langchain",
    "langchain-core",
    "langchain-community",
    "langchain-google-vertexai",
]

for package in packages:
    try:
        print(f"{package}: {version(package)}")
    except PackageNotFoundError:
        print(f"{package}: Not installed")

ragas: 0.4.3
langchain: 1.4.3
langchain-core: 1.6.9
langchain-community: 0.4.2
langchain-google-vertexai: Not installed


In [6]:
# W7D5: Locate the installed Ragas source file

from pathlib import Path
from importlib.metadata import distribution

ragas_path = Path(distribution("ragas").locate_file("ragas/llms/base.py"))

print("Ragas source file:", ragas_path)
print("File exists:", ragas_path.exists())

Ragas source file: c:\Users\ughan\CynarisInternship\AIML_PROJECT\venv\Lib\site-packages\ragas\llms\base.py
File exists: True


In [7]:
# W7D5: Inspect Ragas imports

with ragas_path.open("r", encoding="utf-8") as file:
    lines = file.readlines()

for number, line in enumerate(lines[:25], start=1):
    print(f"{number}: {line.rstrip()}")

1: from __future__ import annotations
2: 
3: import asyncio
4: import inspect
5: import logging
6: import threading
7: import typing as t
8: from abc import ABC, abstractmethod
9: from dataclasses import dataclass, field
10: 
11: import instructor
12: from langchain_community.chat_models.vertexai import ChatVertexAI
13: from langchain_community.llms import VertexAI
14: from langchain_core.language_models import BaseLanguageModel
15: from langchain_core.outputs import ChatGeneration, Generation, LLMResult
16: from langchain_openai.chat_models import AzureChatOpenAI, ChatOpenAI
17: from langchain_openai.llms import AzureOpenAI, OpenAI
18: from langchain_openai.llms.base import BaseOpenAI
19: from pydantic import BaseModel
20: 
21: from ragas._analytics import LLMUsageEvent, track
22: from ragas.cache import CacheInterface, cacher
23: from ragas.exceptions import LLMDidNotFinishException
24: from ragas.run_config import RunConfig, add_async_retry
25: 


In [8]:
# W7D5: Check Vertex AI integration availability

from importlib.util import find_spec

print(
    "Google Vertex AI integration:",
    find_spec("langchain_google_vertexai")
)

Google Vertex AI integration: None


In [9]:
# W7D5: Record Ragas compatibility status

from pathlib import Path

OUTPUT_DIR = Path("notebooks/W7D5/outputs")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

ragas_status = """W7D5 Ragas Compatibility Check

Installed Ragas version: 0.4.3
Issue: Ragas imports a missing langchain_community Vertex AI module.
Replacement integration: Not installed.
Status: Pending compatibility fix and evaluation.
Core RAG pipeline: Working.
"""

status_file = OUTPUT_DIR / "ragas_status.txt"
status_file.write_text(ragas_status, encoding="utf-8")

print("Ragas status recorded.")
print("File:", status_file)

Ragas status recorded.
File: notebooks\W7D5\outputs\ragas_status.txt


In [10]:
# W7D5: Verify project files

from pathlib import Path

project_dir = Path("notebooks/W7D5")

required_files = [
    project_dir / "W7D5_multi_document_rag.ipynb",
    project_dir / "SELF_REVIEW.md",
    project_dir / "outputs/evaluation_results.txt",
    project_dir / "outputs/test_results.txt",
    project_dir / "outputs/interaction_evidence.txt",
    project_dir / "outputs/ragas_status.txt",
]

for file_path in required_files:
    status = "OK" if file_path.exists() else "MISSING"
    print(f"{file_path}: {status}")

notebooks\W7D5\W7D5_multi_document_rag.ipynb: MISSING
notebooks\W7D5\SELF_REVIEW.md: OK
notebooks\W7D5\outputs\evaluation_results.txt: OK
notebooks\W7D5\outputs\test_results.txt: OK
notebooks\W7D5\outputs\interaction_evidence.txt: OK
notebooks\W7D5\outputs\ragas_status.txt: OK


In [11]:
# W7D5: Find the saved notebook

from pathlib import Path

search_dir = Path("notebooks/W7D5")

for file_path in search_dir.rglob("*.ipynb"):
    print("Notebook found:", file_path)

In [12]:
# W7D5: Check the notebook working directory

import os

print("Current working directory:")
print(os.getcwd())

print("\nW7D5 folder exists:")
print(os.path.exists("notebooks/W7D5"))

Current working directory:
c:\Users\ughan\CynarisInternship\AIML_PROJECT\notebooks

W7D5 folder exists:
True


In [13]:
# W7D5: Find the saved notebook

from pathlib import Path

search_dir = Path("W7D5")

print("Folder exists:", search_dir.exists())

for file_path in search_dir.rglob("*.ipynb"):
    print("Notebook found:", file_path)
    

Folder exists: True


In [14]:
# W7D5: Inspect folder contents

from pathlib import Path

folder = Path("W7D5")

for item in folder.iterdir():
    print("Folder:" if item.is_dir() else "File:", item.name)

File: W7D5_multi_document_rag.py


In [15]:
# W7D5: Verify the main Python script

from pathlib import Path

script_path = Path("W7D5/W7D5_multi_document_rag.py")

print("Script exists:", script_path.exists())

if script_path.exists():
    print("Script size:", script_path.stat().st_size, "bytes")

Script exists: True
Script size: 0 bytes
